# cap10_sequence_risk_checkpoint_penalty_screen_v1

Purpose: screen whether a sequence-risk / drawdown-drift checkpoint penalty can select a safer policy from existing `cap10` candidates before opening another PPO training branch.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- Direct comparison baselines: `cap10_joint_contract_checkpoint_drift_selector_v1` and `cap10_sequence_risk_target_cap_bc_screen_v1`.
- Role: `diagnose`; no actor is promoted or retained by this card.
- Target: excess return `>= +30pp` and MaxDD `<= 15%` on each replay fold.
- Changed layer: `model_selection` / checkpoint scoring only. No new profile, no reward code, and no training from this notebook.

Historical reflection:
- Closest success: BC initialization and `trend_break_any_cap10` both preserve enough relative return on the evaluated folds, and `trend_break_any_cap10` improves relative return in both folds.
- Closest failure: prior PPO drift improved validation while worsening `wf_2022` MaxDD to `37%`; direct target caps improved one fold while breaking another.
- Non-duplicate: this run does not add a new cap or PPO run. It reconciles aggregate candidate metrics with available daily paths, then asks if an explicit sequence-risk checkpoint penalty can separate safe policies from high-return / high-drawdown candidates. Aggregate-only candidates receive a conservative missing-path penalty so lack of path evidence cannot be scored as safety.

External basis:
- Qlib-style reproducible research artifacts and workflow separation: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO drift caution / policy update risk: https://arxiv.org/abs/1707.06347
- Financial RL validation and ensemble/checkpoint framing inherited from prior cards: https://arxiv.org/abs/2006.04779 and https://arxiv.org/abs/1712.06924
- Deflated Sharpe / multiple-testing caution inherited from prior cards: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'runtime').exists():
    raise RuntimeError(f'Notebook must run from repo root, got {ROOT}')

EXPERIMENT = 'cap10_sequence_risk_checkpoint_penalty_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CHECKPOINT_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_joint_contract_checkpoint_drift_selector_v1'
TARGET_CAP_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_target_cap_bc_screen_v1'
SEQUENCE_DIAGNOSTIC_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_bc_sequence_risk_state_diagnostic_v1'

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0
DRIFT_MAX_DD_CEILING_PP = 2.0
DRIFT_RELATIVE_FLOOR_PP = -5.0
SEQUENCE_POSITION_REFERENCE = 0.10
MISSING_SEQUENCE_PATH_PENALTY = 0.35

CARD.mkdir(parents=True, exist_ok=True)
print('card', CARD)


card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/cap10_sequence_risk_checkpoint_penalty_screen_v1


In [2]:
checkpoint_selection = json.loads((CHECKPOINT_CARD / 'selection.json').read_text())
target_cap_selection = json.loads((TARGET_CAP_CARD / 'selection.json').read_text())
sequence_selection = json.loads((SEQUENCE_DIAGNOSTIC_CARD / 'selection.json').read_text())

checkpoint_metrics = pd.read_csv(CHECKPOINT_CARD / 'checkpoint_candidate_metrics.csv')
target_cap_metrics = pd.read_csv(TARGET_CAP_CARD / 'bc_candidate_metrics.csv')
target_cap_summary = pd.read_csv(TARGET_CAP_CARD / 'candidate_summary.csv')
sequence_cap_audit = pd.read_csv(TARGET_CAP_CARD / 'sequence_risk_cap_audit.csv')
risk_state_screen = pd.read_csv(SEQUENCE_DIAGNOSTIC_CARD / 'risk_state_candidate_screen.csv')

source_context = {
    'checkpoint_decision': checkpoint_selection.get('decision'),
    'target_cap_decision': target_cap_selection.get('decision'),
    'sequence_diagnostic_decision': sequence_selection.get('decision'),
    'target_cap_selected_candidate': target_cap_selection.get('selected_candidate'),
    'target_cap_robust_improvement_candidate_count': target_cap_selection.get('robust_improvement_candidate_count'),
}
source_context


{'checkpoint_decision': 'checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion',
 'target_cap_decision': 'sequence_risk_target_cap_bc_no_robust_improvement_no_promotion',
 'sequence_diagnostic_decision': 'sequence_risk_state_candidate_ready_for_target_screen_no_promotion',
 'target_cap_selected_candidate': 'baseline_no_sequence_cap',
 'target_cap_robust_improvement_candidate_count': 0}

In [3]:
def as_float(value: Any, default: float = float('nan')) -> float:
    try:
        if value is None:
            return default
        out = float(value)
        if math.isnan(out):
            return default
        return out
    except Exception:
        return default


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return None if not math.isfinite(value) else value
    if pd.isna(value) if not isinstance(value, (dict, list, tuple)) else False:
        return None
    return value


def parse_fold_from_train_fold(value: str) -> str:
    value = str(value)
    if value.endswith('_train'):
        return value[:-6]
    return value


def target_gate(relative_return_pp: float, max_dd_pct: float) -> bool:
    return bool(relative_return_pp >= TARGET_RELATIVE_RETURN_PP and max_dd_pct <= TARGET_MAX_DD_PCT)


def gate_loss(relative_return_pp: float, max_dd_pct: float) -> float:
    relative_gap = max(0.0, TARGET_RELATIVE_RETURN_PP - relative_return_pp)
    dd_gap = max(0.0, max_dd_pct - TARGET_MAX_DD_PCT)
    return relative_gap / TARGET_RELATIVE_RETURN_PP + 2.0 * dd_gap / TARGET_MAX_DD_PCT


def drift_loss(row: pd.Series) -> float:
    max_dd_delta = as_float(row.get('max_dd_delta_vs_bc'), 0.0)
    relative_delta = as_float(row.get('relative_delta_vs_bc'), 0.0)
    dd_component = max(0.0, max_dd_delta - DRIFT_MAX_DD_CEILING_PP) / TARGET_MAX_DD_PCT
    relative_component = max(0.0, DRIFT_RELATIVE_FLOOR_PP - relative_delta) / TARGET_RELATIVE_RETURN_PP
    return dd_component + relative_component


def normalize_checkpoint_rows(df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        relative_return = as_float(row.get('test_relative_return'))
        max_dd = as_float(row.get('test_max_dd'))
        candidate = str(row['candidate'])
        rows.append({
            'source': 'checkpoint_selector',
            'candidate': candidate,
            'candidate_id': f'checkpoint_selector::{candidate}',
            'candidate_kind': str(row.get('candidate_kind', 'checkpoint')),
            'fold': str(row['fold']),
            'reported_return_pct': as_float(row.get('test_return')),
            'reported_sharpe': as_float(row.get('test_sharpe')),
            'reported_max_dd_pct': max_dd,
            'reported_avg_pos': as_float(row.get('test_avg_pos')),
            'reported_buyhold_return_pct': as_float(row.get('test_buy_and_hold_return')),
            'reported_relative_return_pp': relative_return,
            'valid_return_pct': as_float(row.get('valid_return')),
            'valid_sharpe': as_float(row.get('valid_sharpe')),
            'valid_max_dd_pct': as_float(row.get('valid_max_dd')),
            'relative_delta_vs_bc_pp': as_float(row.get('relative_delta_vs_bc'), 0.0),
            'max_dd_delta_vs_bc_pp': as_float(row.get('max_dd_delta_vs_bc'), 0.0),
            'complexity': 0,
            'bc_slice_gate': bool(row.get('bc_slice_gate', False)),
            'reported_target_gate': target_gate(relative_return, max_dd),
            'reported_drift_gate': bool(row.get('drift_gate', False)),
            'aggregate_metric_source': 'cap10_joint_contract_checkpoint_drift_selector_v1/checkpoint_candidate_metrics.csv',
        })
    return pd.DataFrame(rows)


def normalize_target_cap_rows(df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        relative_return = as_float(row.get('test_relative_return'))
        max_dd = as_float(row.get('test_max_dd'))
        candidate = str(row['candidate'])
        rows.append({
            'source': 'sequence_target_cap_bc',
            'candidate': candidate,
            'candidate_id': f'sequence_target_cap_bc::{candidate}',
            'candidate_kind': 'bc_sequence_target_cap',
            'fold': str(row['fold']),
            'reported_return_pct': as_float(row.get('test_return')),
            'reported_sharpe': as_float(row.get('test_sharpe')),
            'reported_max_dd_pct': max_dd,
            'reported_avg_pos': as_float(row.get('test_avg_pos')),
            'reported_buyhold_return_pct': as_float(row.get('test_buy_and_hold_return')),
            'reported_relative_return_pp': relative_return,
            'valid_return_pct': as_float(row.get('valid_return')),
            'valid_sharpe': as_float(row.get('valid_sharpe')),
            'valid_max_dd_pct': as_float(row.get('valid_max_dd')),
            'relative_delta_vs_bc_pp': 0.0,
            'max_dd_delta_vs_bc_pp': 0.0,
            'complexity': int(as_float(row.get('complexity'), 0.0)),
            'bc_slice_gate': bool(row.get('bc_slice_gate', False)),
            'reported_target_gate': target_gate(relative_return, max_dd),
            'reported_drift_gate': True,
            'aggregate_metric_source': 'cap10_sequence_risk_target_cap_bc_screen_v1/bc_candidate_metrics.csv',
        })
    return pd.DataFrame(rows)

fold_metrics = pd.concat(
    [normalize_checkpoint_rows(checkpoint_metrics), normalize_target_cap_rows(target_cap_metrics)],
    ignore_index=True,
)
fold_metrics['reported_gate_loss'] = [
    gate_loss(rel, dd)
    for rel, dd in zip(fold_metrics['reported_relative_return_pp'], fold_metrics['reported_max_dd_pct'])
]
fold_metrics['reported_drift_loss'] = [
    drift_loss(row)
    for _, row in fold_metrics.iterrows()
]
fold_metrics.head(12)


,source,candidate,candidate_id,candidate_kind,fold,reported_return_pct,reported_sharpe,reported_max_dd_pct,reported_avg_pos,reported_buyhold_return_pct,...,valid_max_dd_pct,relative_delta_vs_bc_pp,max_dd_delta_vs_bc_pp,complexity,bc_slice_gate,reported_target_gate,reported_drift_gate,aggregate_metric_source,reported_gate_loss,reported_drift_loss
0,checkpoint_selector,bc_initialization,checkpoint_selector::bc_initialization,bc_initialization,wf_2022,-11.326201,-0.371798,18.455088,0.280667,-76.614257,...,9.424797,0.000000,0.000000,0,True,False,True,cap10_joint_contract_checkpoint_drift_selector...,0.460678,0.0
1,checkpoint_selector,ppo_retained_iter_0_bc_equivalent,checkpoint_selector::ppo_retained_iter_0_bc_eq...,ppo_retained,wf_2022,-11.326201,-0.371798,18.455088,0.280667,-76.614257,...,12.185863,0.000000,0.000000,0,True,False,True,cap10_joint_contract_checkpoint_drift_selector...,0.460678,0.0
2,checkpoint_selector,ppo_retained_iter_4,checkpoint_selector::ppo_retained_iter_4,ppo_retained,wf_2022,-29.598687,-0.776461,37.275221,0.318036,-76.614257,...,15.892958,5.351209,18.820132,0,True,False,False,cap10_joint_contract_checkpoint_drift_selector...,2.970029,0.0
3,checkpoint_selector,ppo_selected_best,checkpoint_selector::ppo_selected_best,ppo_selected,wf_2022,-29.598687,-0.776461,37.275221,0.318036,-76.614257,...,15.892958,5.351209,18.820132,0,True,False,False,cap10_joint_contract_checkpoint_drift_selector...,2.970029,0.0
4,checkpoint_selector,bc_initialization,checkpoint_selector::bc_initialization,bc_initialization,wf_2023_2024,-7.110714,-0.429241,13.597475,0.174542,-39.952710,...,11.572674,0.000000,0.000000,0,True,True,True,cap10_joint_contract_checkpoint_drift_selector...,0.000000,0.0
5,checkpoint_selector,ppo_retained_iter_0_bc_equivalent,checkpoint_selector::ppo_retained_iter_0_bc_eq...,ppo_retained,wf_2023_2024,-7.110714,-0.429241,13.597475,0.174542,-39.952710,...,11.509663,0.000000,0.000000,0,True,True,True,cap10_joint_contract_checkpoint_drift_selector...,0.000000,0.0
6,checkpoint_selector,ppo_retained_iter_4,checkpoint_selector::ppo_retained_iter_4,ppo_retained,wf_2023_2024,-6.024125,-0.423117,14.652352,0.150589,-39.952710,...,12.292481,14.312349,1.054878,0,True,True,True,cap10_joint_contract_checkpoint_drift_selector...,0.000000,0.0
7,checkpoint_selector,ppo_selected_best,checkpoint_selector::ppo_selected_best,ppo_selected,wf_2023_2024,-6.024125,-0.423117,14.652352,0.150589,-39.952710,...,11.509663,14.312349,1.054878,0,True,True,True,cap10_joint_contract_checkpoint_drift_selector...,0.000000,0.0
8,sequence_target_cap_bc,baseline_no_sequence_cap,sequence_target_cap_bc::baseline_no_sequence_cap,bc_sequence_target_cap,wf_2022,-16.534504,-0.804593,18.396429,0.280667,-76.614257,...,10.528546,0.000000,0.000000,0,True,False,True,cap10_sequence_risk_target_cap_bc_screen_v1/bc...,0.452857,0.0
9,sequence_target_cap_bc,baseline_no_sequence_cap,sequence_target_cap_bc::baseline_no_sequence_cap,bc_sequence_target_cap,wf_2023_2024,-5.811867,-0.355173,13.472617,0.174542,-39.952710,...,20.436342,0.000000,0.000000,0,True,True,True,cap10_sequence_risk_target_cap_bc_screen_v1/bc...,0.000000,0.0


In [4]:
def filename_candidate_fold(path: Path) -> tuple[str, str] | None:
    name = path.name
    if not name.endswith('_daily_path.csv'):
        return None
    stem = name.removesuffix('_daily_path.csv')
    marker = '_wf_'
    if marker not in stem:
        return None
    candidate, fold_tail = stem.split(marker, 1)
    return candidate, f'wf_{fold_tail}'


def normalized_drawdown_pct(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce').fillna(0.0)
    if len(values) and float(values.abs().max()) <= 1.5:
        values = values * 100.0
    return values


def nav_return_pct(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors='coerce').dropna()
    if len(values) < 2 or values.iloc[0] == 0:
        return float('nan')
    return float((values.iloc[-1] / values.iloc[0] - 1.0) * 100.0)


def load_daily_paths() -> dict[tuple[str, str], Path]:
    paths: dict[tuple[str, str], Path] = {}
    for path in sorted(TARGET_CAP_CARD.glob('*_daily_path.csv')):
        parsed = filename_candidate_fold(path)
        if parsed:
            paths[parsed] = path
    diagnostic_path = SEQUENCE_DIAGNOSTIC_CARD / 'bc_wf2022_daily_path.csv'
    if diagnostic_path.exists():
        paths[('bc_initialization_official_daily_replay', 'wf_2022')] = diagnostic_path
    return paths


def thresholds_by_fold(candidate: str = 'trend_break_any_cap10') -> dict[str, dict[str, Any]]:
    out: dict[str, dict[str, Any]] = {}
    subset = sequence_cap_audit[sequence_cap_audit['candidate'].eq(candidate)]
    for _, row in subset.iterrows():
        thresholds = json.loads(str(row.get('thresholds_json') or '{}'))
        out[parse_fold_from_train_fold(str(row['fold']))] = {
            'thresholds': thresholds,
            'mode': str(row.get('mode', 'any')),
            'source_fold': str(row['fold']),
            'candidate': candidate,
        }
    return out


def apply_sequence_thresholds(df: pd.DataFrame, spec: dict[str, Any]) -> tuple[pd.Series, pd.Series]:
    thresholds = spec.get('thresholds', {}) if spec else {}
    mode = spec.get('mode', 'any') if spec else 'any'
    if not thresholds:
        empty = pd.Series(False, index=df.index)
        return empty, pd.Series(0, index=df.index, dtype=int)
    hits = []
    for feature, threshold_spec in thresholds.items():
        if feature not in df.columns:
            hits.append(pd.Series(False, index=df.index))
            continue
        threshold = as_float(threshold_spec.get('threshold'))
        direction = str(threshold_spec.get('direction', 'low'))
        values = pd.to_numeric(df[feature], errors='coerce')
        if not math.isfinite(threshold):
            hit = pd.Series(False, index=df.index)
        elif direction == 'high':
            hit = values >= threshold
        else:
            hit = values <= threshold
        hits.append(hit.fillna(False).astype(bool))
    hit_frame = pd.concat(hits, axis=1) if hits else pd.DataFrame(index=df.index)
    hit_count = hit_frame.sum(axis=1).astype(int) if not hit_frame.empty else pd.Series(0, index=df.index, dtype=int)
    if mode == 'two_of_three':
        mask = hit_count >= 2
    elif mode == 'all':
        mask = hit_count >= len(thresholds)
    elif mode == 'none':
        mask = pd.Series(False, index=df.index)
    else:
        mask = hit_count >= 1
    return mask.astype(bool), hit_count


def daily_path_metrics(candidate: str, fold: str, path: Path, threshold_specs: dict[str, dict[str, Any]]) -> dict[str, Any]:
    df = pd.read_csv(path)
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'])
    drawdown_pct = normalized_drawdown_pct(df.get('drawdown', pd.Series(0.0, index=df.index)))
    position = pd.to_numeric(df.get('position', pd.Series(np.nan, index=df.index)), errors='coerce')
    strategy_return = nav_return_pct(df['strategy_nav'] if 'strategy_nav' in df.columns else df['nav'])
    buyhold_return = nav_return_pct(df['buyhold_nav']) if 'buyhold_nav' in df.columns else float('nan')
    spec = threshold_specs.get(fold, {})
    sequence_mask, sequence_hit_count = apply_sequence_thresholds(df, spec)
    breach_mask = drawdown_pct > TARGET_MAX_DD_PCT
    sequence_rows = int(sequence_mask.sum())
    breach_rows = int(breach_mask.sum())
    sequence_breach_rows = int((sequence_mask & breach_mask).sum())
    sequence_avg_pos = float(position[sequence_mask].mean()) if sequence_rows else 0.0
    non_sequence_avg_pos = float(position[~sequence_mask].mean()) if sequence_rows < len(position) else 0.0
    sequence_max_dd = float(drawdown_pct[sequence_mask].max()) if sequence_rows else 0.0
    sequence_breach_capture = sequence_breach_rows / breach_rows if breach_rows else 0.0
    sequence_share = sequence_rows / max(len(df), 1)
    sequence_exposure_loss = max(0.0, sequence_avg_pos - SEQUENCE_POSITION_REFERENCE) / max(SEQUENCE_POSITION_REFERENCE, 1e-9)
    sequence_drawdown_loss = max(0.0, sequence_max_dd - TARGET_MAX_DD_PCT) / TARGET_MAX_DD_PCT
    sequence_breach_loss = sequence_breach_capture * sequence_exposure_loss
    path_penalty = 0.35 * sequence_exposure_loss + 0.40 * sequence_drawdown_loss + 0.25 * sequence_breach_loss
    return {
        'candidate': candidate,
        'fold': fold,
        'daily_path': str(path.relative_to(ROOT)),
        'rows': int(len(df)),
        'start': str(df['datetime'].min().date()) if 'datetime' in df.columns and len(df) else None,
        'end': str(df['datetime'].max().date()) if 'datetime' in df.columns and len(df) else None,
        'daily_return_pct': strategy_return,
        'daily_buyhold_return_pct': buyhold_return,
        'daily_relative_return_pp': strategy_return - buyhold_return if math.isfinite(strategy_return) and math.isfinite(buyhold_return) else float('nan'),
        'daily_max_dd_pct': float(drawdown_pct.max()) if len(drawdown_pct) else float('nan'),
        'daily_avg_pos': float(position.mean()) if len(position) else float('nan'),
        'threshold_source_candidate': spec.get('candidate'),
        'threshold_source_fold': spec.get('source_fold'),
        'threshold_mode': spec.get('mode'),
        'thresholds_json': json.dumps(json_safe(spec.get('thresholds', {})), sort_keys=True),
        'sequence_rows': sequence_rows,
        'sequence_share': sequence_share,
        'sequence_avg_pos': sequence_avg_pos,
        'non_sequence_avg_pos': non_sequence_avg_pos,
        'sequence_position_lift': sequence_avg_pos - non_sequence_avg_pos,
        'sequence_max_dd_pct': sequence_max_dd,
        'drawdown_breach_rows': breach_rows,
        'sequence_breach_rows': sequence_breach_rows,
        'sequence_breach_capture': sequence_breach_capture,
        'sequence_hit_count_mean': float(sequence_hit_count.mean()) if len(sequence_hit_count) else 0.0,
        'sequence_path_penalty': float(path_penalty),
    }

threshold_specs = thresholds_by_fold('trend_break_any_cap10')
daily_rows = []
for (candidate, fold), path in load_daily_paths().items():
    daily_rows.append(daily_path_metrics(candidate, fold, path, threshold_specs))

daily_sequence_risk = pd.DataFrame(daily_rows).sort_values(['candidate', 'fold']).reset_index(drop=True)
daily_sequence_risk.to_csv(CARD / 'daily_sequence_risk_penalty.csv', index=False)
daily_sequence_risk


,candidate,fold,daily_path,rows,start,end,daily_return_pct,daily_buyhold_return_pct,daily_relative_return_pp,daily_max_dd_pct,...,sequence_share,sequence_avg_pos,non_sequence_avg_pos,sequence_position_lift,sequence_max_dd_pct,drawdown_breach_rows,sequence_breach_rows,sequence_breach_capture,sequence_hit_count_mean,sequence_path_penalty
0,baseline_no_sequence_cap,wf_2022,runtime/model_selection/card/cap10_sequence_ri...,250,2022-01-03,2022-12-29,-16.534504,-86.588072,70.053568,18.396429,...,0.9,0.160863,0.042335,0.118528,18.396429,30,30,1.0,1.676,0.455747
1,baseline_no_sequence_cap,wf_2023_2024,runtime/model_selection/card/cap10_sequence_ri...,251,2024-01-02,2024-12-30,-5.811867,-0.356635,-5.455232,13.472617,...,0.0,0.000000,0.145913,-0.145913,0.000000,0,0,0.0,0.000,0.000000
2,bc_initialization_official_daily_replay,wf_2022,runtime/model_selection/card/cap10_bc_sequence...,250,2022-01-03,2022-12-29,-16.534504,-86.588072,70.053568,18.396429,...,0.9,0.160863,0.042335,0.118528,18.396429,30,30,1.0,1.676,0.455747
3,trend_break_any_cap10,wf_2022,runtime/model_selection/card/cap10_sequence_ri...,250,2022-01-03,2022-12-29,-18.062932,-86.588072,68.525139,19.550984,...,0.9,0.135131,0.042335,0.092796,19.550984,46,46,1.0,1.676,0.332146
4,trend_break_any_cap10,wf_2023_2024,runtime/model_selection/card/cap10_sequence_ri...,251,2024-01-02,2024-12-30,-5.841762,-0.356635,-5.485127,11.338372,...,0.0,0.000000,0.149038,-0.149038,0.000000,0,0,0.0,0.000,0.000000


In [5]:
# Reconcile source-reported aggregate metrics with recomputed daily paths when paths are available.
daily_for_merge = daily_sequence_risk.rename(columns={
    'daily_return_pct': 'path_return_pct',
    'daily_buyhold_return_pct': 'path_buyhold_return_pct',
    'daily_relative_return_pp': 'path_relative_return_pp',
    'daily_max_dd_pct': 'path_max_dd_pct',
    'daily_avg_pos': 'path_avg_pos',
})

reconciliation = fold_metrics.merge(
    daily_for_merge[[
        'candidate', 'fold', 'daily_path', 'path_return_pct', 'path_buyhold_return_pct',
        'path_relative_return_pp', 'path_max_dd_pct', 'path_avg_pos',
        'sequence_rows', 'sequence_share', 'sequence_avg_pos', 'sequence_max_dd_pct',
        'sequence_breach_capture', 'sequence_path_penalty'
    ]],
    on=['candidate', 'fold'],
    how='left',
)
reconciliation['daily_path_available'] = reconciliation['daily_path'].notna()
for reported_col, path_col, out_col in [
    ('reported_return_pct', 'path_return_pct', 'return_delta_reported_minus_path'),
    ('reported_buyhold_return_pct', 'path_buyhold_return_pct', 'buyhold_delta_reported_minus_path'),
    ('reported_relative_return_pp', 'path_relative_return_pp', 'relative_delta_reported_minus_path'),
    ('reported_max_dd_pct', 'path_max_dd_pct', 'max_dd_delta_reported_minus_path'),
    ('reported_avg_pos', 'path_avg_pos', 'avg_pos_delta_reported_minus_path'),
]:
    reconciliation[out_col] = reconciliation[reported_col] - reconciliation[path_col]

reconciliation.to_csv(CARD / 'metric_source_reconciliation.csv', index=False)
reconciliation[
    ['source', 'candidate', 'fold', 'reported_relative_return_pp', 'path_relative_return_pp',
     'reported_max_dd_pct', 'path_max_dd_pct', 'daily_path_available', 'relative_delta_reported_minus_path']
].sort_values(['source', 'candidate', 'fold']).head(30)


,source,candidate,fold,reported_relative_return_pp,path_relative_return_pp,reported_max_dd_pct,path_max_dd_pct,daily_path_available,relative_delta_reported_minus_path
0,checkpoint_selector,bc_initialization,wf_2022,36.419124,NaN,18.455088,NaN,False,NaN
4,checkpoint_selector,bc_initialization,wf_2023_2024,30.501116,NaN,13.597475,NaN,False,NaN
1,checkpoint_selector,ppo_retained_iter_0_bc_equivalent,wf_2022,36.419124,NaN,18.455088,NaN,False,NaN
5,checkpoint_selector,ppo_retained_iter_0_bc_equivalent,wf_2023_2024,30.501116,NaN,13.597475,NaN,False,NaN
2,checkpoint_selector,ppo_retained_iter_4,wf_2022,41.770333,NaN,37.275221,NaN,False,NaN
6,checkpoint_selector,ppo_retained_iter_4,wf_2023_2024,44.813465,NaN,14.652352,NaN,False,NaN
3,checkpoint_selector,ppo_selected_best,wf_2022,41.770333,NaN,37.275221,NaN,False,NaN
7,checkpoint_selector,ppo_selected_best,wf_2023_2024,44.813465,NaN,14.652352,NaN,False,NaN
12,sequence_target_cap_bc,anchor_any_cap15,wf_2022,57.616373,NaN,24.884684,NaN,False,NaN
13,sequence_target_cap_bc,anchor_any_cap15,wf_2023_2024,37.183853,NaN,22.704458,NaN,False,NaN


In [6]:
scored = reconciliation.copy()
scored['sequence_path_penalty'] = pd.to_numeric(scored['sequence_path_penalty'], errors='coerce')
scored['sequence_path_available'] = scored['sequence_path_penalty'].notna()
scored['missing_sequence_path_penalty'] = np.where(scored['sequence_path_available'], 0.0, MISSING_SEQUENCE_PATH_PENALTY)
scored['complexity_penalty'] = 0.02 * pd.to_numeric(scored['complexity'], errors='coerce').fillna(0.0)
scored['sequence_path_penalty_for_score'] = scored['sequence_path_penalty'].fillna(0.0)
scored['checkpoint_penalty_loss'] = (
    scored['reported_gate_loss']
    + scored['reported_drift_loss']
    + scored['sequence_path_penalty_for_score']
    + scored['missing_sequence_path_penalty']
    + scored['complexity_penalty']
)
scored['reported_full_target_gate'] = scored['reported_target_gate']
scored['sequence_path_status'] = np.where(scored['sequence_path_available'], 'daily_path_available', 'aggregate_only')

fold_score_columns = [
    'source', 'candidate', 'candidate_id', 'candidate_kind', 'fold',
    'reported_relative_return_pp', 'reported_max_dd_pct', 'reported_avg_pos',
    'reported_target_gate', 'reported_gate_loss', 'reported_drift_loss',
    'sequence_path_status', 'sequence_rows', 'sequence_share', 'sequence_avg_pos',
    'sequence_max_dd_pct', 'sequence_breach_capture', 'sequence_path_penalty',
    'missing_sequence_path_penalty', 'complexity_penalty', 'checkpoint_penalty_loss',
    'aggregate_metric_source', 'daily_path',
]
fold_penalty_scores = scored[fold_score_columns].sort_values(['checkpoint_penalty_loss', 'source', 'candidate', 'fold']).reset_index(drop=True)
fold_penalty_scores.to_csv(CARD / 'fold_penalty_scores.csv', index=False)
fold_penalty_scores.head(30)


,source,candidate,candidate_id,candidate_kind,fold,reported_relative_return_pp,reported_max_dd_pct,reported_avg_pos,reported_target_gate,reported_gate_loss,...,sequence_share,sequence_avg_pos,sequence_max_dd_pct,sequence_breach_capture,sequence_path_penalty,missing_sequence_path_penalty,complexity_penalty,checkpoint_penalty_loss,aggregate_metric_source,daily_path
0,sequence_target_cap_bc,baseline_no_sequence_cap,sequence_target_cap_bc::baseline_no_sequence_cap,bc_sequence_target_cap,wf_2023_2024,30.501116,13.472617,0.174542,True,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.00,0.00,0.000000,cap10_sequence_risk_target_cap_bc_screen_v1/bc...,runtime/model_selection/card/cap10_sequence_ri...
1,sequence_target_cap_bc,trend_break_any_cap10,sequence_target_cap_bc::trend_break_any_cap10,bc_sequence_target_cap,wf_2023_2024,37.779333,11.338372,0.151505,True,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.00,0.08,0.080000,cap10_sequence_risk_target_cap_bc_screen_v1/bc...,runtime/model_selection/card/cap10_sequence_ri...
2,checkpoint_selector,bc_initialization,checkpoint_selector::bc_initialization,bc_initialization,wf_2023_2024,30.501116,13.597475,0.174542,True,0.000000,...,NaN,NaN,NaN,NaN,NaN,0.35,0.00,0.350000,cap10_joint_contract_checkpoint_drift_selector...,NaN
3,checkpoint_selector,ppo_retained_iter_0_bc_equivalent,checkpoint_selector::ppo_retained_iter_0_bc_eq...,ppo_retained,wf_2023_2024,30.501116,13.597475,0.174542,True,0.000000,...,NaN,NaN,NaN,NaN,NaN,0.35,0.00,0.350000,cap10_joint_contract_checkpoint_drift_selector...,NaN
4,checkpoint_selector,ppo_retained_iter_4,checkpoint_selector::ppo_retained_iter_4,ppo_retained,wf_2023_2024,44.813465,14.652352,0.150589,True,0.000000,...,NaN,NaN,NaN,NaN,NaN,0.35,0.00,0.350000,cap10_joint_contract_checkpoint_drift_selector...,NaN
5,checkpoint_selector,ppo_selected_best,checkpoint_selector::ppo_selected_best,ppo_selected,wf_2023_2024,44.813465,14.652352,0.150589,True,0.000000,...,NaN,NaN,NaN,NaN,NaN,0.35,0.00,0.350000,cap10_joint_contract_checkpoint_drift_selector...,NaN
6,sequence_target_cap_bc,trend_break_any_cap15,sequence_target_cap_bc::trend_break_any_cap15,bc_sequence_target_cap,wf_2023_2024,39.711227,9.805398,0.155236,True,0.000000,...,NaN,NaN,NaN,NaN,NaN,0.35,0.06,0.410000,cap10_sequence_risk_target_cap_bc_screen_v1/bc...,NaN
7,sequence_target_cap_bc,slope_low_cap15,sequence_target_cap_bc::slope_low_cap15,bc_sequence_target_cap,wf_2022,59.461195,16.115577,0.188882,False,0.148744,...,NaN,NaN,NaN,NaN,NaN,0.35,0.02,0.518744,cap10_sequence_risk_target_cap_bc_screen_v1/bc...,NaN
8,checkpoint_selector,bc_initialization,checkpoint_selector::bc_initialization,bc_initialization,wf_2022,36.419124,18.455088,0.280667,False,0.460678,...,NaN,NaN,NaN,NaN,NaN,0.35,0.00,0.810678,cap10_joint_contract_checkpoint_drift_selector...,NaN
9,checkpoint_selector,ppo_retained_iter_0_bc_equivalent,checkpoint_selector::ppo_retained_iter_0_bc_eq...,ppo_retained,wf_2022,36.419124,18.455088,0.280667,False,0.460678,...,NaN,NaN,NaN,NaN,NaN,0.35,0.00,0.810678,cap10_joint_contract_checkpoint_drift_selector...,NaN


In [7]:
def summarize_candidate(group: pd.DataFrame) -> pd.Series:
    fold_count = int(group['fold'].nunique())
    target_pass_count = int(group['reported_target_gate'].sum())
    daily_path_fold_count = int(group['sequence_path_available'].sum())
    min_relative = float(group['reported_relative_return_pp'].min())
    max_dd = float(group['reported_max_dd_pct'].max())
    full_gate = bool(fold_count >= 2 and target_pass_count == fold_count and min_relative >= TARGET_RELATIVE_RETURN_PP and max_dd <= TARGET_MAX_DD_PCT)
    sequence_ready = bool(daily_path_fold_count == fold_count)
    return pd.Series({
        'source': group['source'].iloc[0],
        'candidate': group['candidate'].iloc[0],
        'candidate_kind': group['candidate_kind'].iloc[0],
        'fold_count': fold_count,
        'target_pass_count': target_pass_count,
        'daily_path_fold_count': daily_path_fold_count,
        'sequence_path_ready': sequence_ready,
        'full_gate': full_gate,
        'min_relative_return_pp': min_relative,
        'max_max_dd_pct': max_dd,
        'mean_avg_pos': float(group['reported_avg_pos'].mean()),
        'max_sequence_share': float(pd.to_numeric(group['sequence_share'], errors='coerce').max()) if group['sequence_share'].notna().any() else 0.0,
        'max_sequence_avg_pos': float(pd.to_numeric(group['sequence_avg_pos'], errors='coerce').max()) if group['sequence_avg_pos'].notna().any() else 0.0,
        'max_sequence_max_dd_pct': float(pd.to_numeric(group['sequence_max_dd_pct'], errors='coerce').max()) if group['sequence_max_dd_pct'].notna().any() else 0.0,
        'max_sequence_breach_capture': float(pd.to_numeric(group['sequence_breach_capture'], errors='coerce').max()) if group['sequence_breach_capture'].notna().any() else 0.0,
        'mean_gate_loss': float(group['reported_gate_loss'].mean()),
        'mean_drift_loss': float(group['reported_drift_loss'].mean()),
        'mean_sequence_path_penalty': float(group['sequence_path_penalty_for_score'].mean()),
        'mean_missing_sequence_path_penalty': float(group['missing_sequence_path_penalty'].mean()),
        'mean_complexity_penalty': float(group['complexity_penalty'].mean()),
        'checkpoint_penalty_loss': float(group['checkpoint_penalty_loss'].mean()),
        'worst_fold_loss': float(group['checkpoint_penalty_loss'].max()),
    })

candidate_penalty_scores = (
    scored.groupby('candidate_id', sort=False)
    .apply(summarize_candidate, include_groups=False)
    .reset_index()
    .sort_values(['full_gate', 'checkpoint_penalty_loss'], ascending=[False, True])
    .reset_index(drop=True)
)

eligible = candidate_penalty_scores[candidate_penalty_scores['fold_count'] >= 2].copy()
selected = eligible.sort_values(['full_gate', 'checkpoint_penalty_loss', 'worst_fold_loss']).iloc[0].to_dict() if len(eligible) else {}
full_gate_candidates = eligible[eligible['full_gate']]
robust_daily_candidates = full_gate_candidates[full_gate_candidates['sequence_path_ready']]

if robust_daily_candidates.empty and selected.get('candidate') in {'baseline_no_sequence_cap', 'bc_initialization', 'ppo_retained_iter_0_bc_equivalent'}:
    decision = 'sequence_risk_checkpoint_penalty_confirms_no_safe_policy_improvement_no_promotion'
    next_mechanism = 'implement_or_screen_reusable_reward_drawdown_drift_penalty_or_sequence_risk_state_feature_before_longer_ppo'
elif not robust_daily_candidates.empty:
    decision = 'sequence_risk_checkpoint_penalty_candidate_ready_for_drift_selector_no_promotion'
    next_mechanism = 'rerun_checkpoint_drift_selector_with_daily_paths_for_selected_candidate_before_any_promotion'
else:
    decision = 'sequence_risk_checkpoint_penalty_insufficient_daily_safe_candidate_no_promotion'
    next_mechanism = 'collect_daily_paths_or_train_reward_with_sequence_risk_feature_before_longer_ppo'

candidate_penalty_scores.to_csv(CARD / 'candidate_penalty_scores.csv', index=False)

selection = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'experiment': EXPERIMENT,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'cap10_joint_contract_checkpoint_drift_selector_v1',
        'cap10_sequence_risk_target_cap_bc_screen_v1',
    ],
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'checkpoint_selection'],
    'profile': target_cap_selection.get('profile', checkpoint_selection.get('profile')),
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'decision': decision,
    'selected_candidate_id': selected.get('candidate_id'),
    'selected_candidate': selected.get('candidate'),
    'selected_source': selected.get('source'),
    'selected_full_gate': bool(selected.get('full_gate', False)),
    'selected_sequence_path_ready': bool(selected.get('sequence_path_ready', False)),
    'selected_min_relative_return_pp': selected.get('min_relative_return_pp'),
    'selected_max_max_dd_pct': selected.get('max_max_dd_pct'),
    'selected_checkpoint_penalty_loss': selected.get('checkpoint_penalty_loss'),
    'full_gate_candidate_count': int(len(full_gate_candidates)),
    'robust_daily_full_gate_candidate_count': int(len(robust_daily_candidates)),
    'aggregate_candidate_count': int(len(candidate_penalty_scores)),
    'daily_path_candidate_fold_count': int(scored['sequence_path_available'].sum()),
    'metric_reconciliation_note': 'Target gate and checkpoint score use source-reported aggregate metrics for continuity; available daily paths are recomputed only for sequence-risk/path diagnostics because prior cards used mixed buy-and-hold baselines. Aggregate-only candidates receive a conservative missing-path penalty.',
    'anti_leakage_note': 'Sequence-risk path flags use fold-local train thresholds inherited from the target-cap audit and are applied to evaluation daily paths for scoring only; no OOS thresholds become training labels here.',
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'promotion_blocker': 'No candidate passes both replay folds with +30pp relative return and <=15% MaxDD; this is a diagnostic checkpoint screen only.',
    'next_mechanism': next_mechanism,
    'source_decisions': source_context,
    'external_sources': target_cap_selection.get('external_sources', checkpoint_selection.get('external_sources', [])),
    'sidecars': [
        'daily_sequence_risk_penalty.csv',
        'metric_source_reconciliation.csv',
        'fold_penalty_scores.csv',
        'candidate_penalty_scores.csv',
        'selection.json',
        'summary.json',
    ],
}

summary = {
    'decision': decision,
    'selection': selection,
    'selected_candidate': json_safe(selected),
    'candidate_penalty_scores': json_safe(candidate_penalty_scores.to_dict(orient='records')),
    'fold_penalty_scores': json_safe(fold_penalty_scores.to_dict(orient='records')),
    'daily_sequence_risk_penalty': json_safe(daily_sequence_risk.to_dict(orient='records')),
    'source_context': source_context,
    'conclusion': [
        f"Selected {selected.get('candidate_id')} with loss={selected.get('checkpoint_penalty_loss'):.4f}, full_gate={bool(selected.get('full_gate', False))}.",
        f"Selected max MaxDD={selected.get('max_max_dd_pct'):.2f}% and min relative={selected.get('min_relative_return_pp'):.2f}pp versus target <=15% / >=30pp.",
        f"Full-gate candidates={len(full_gate_candidates)}, robust daily full-gate candidates={len(robust_daily_candidates)}.",
        'No runtime promotion, no actor retention, and no longer PPO branch from this diagnostic card.',
    ] if selected else ['No eligible candidate with two folds was available.'],
}

(CARD / 'selection.json').write_text(json.dumps(json_safe(selection), indent=2, sort_keys=True) + '\n')
(CARD / 'summary.json').write_text(json.dumps(json_safe(summary), indent=2, sort_keys=True) + '\n')

candidate_penalty_scores


,candidate_id,source,candidate,candidate_kind,fold_count,target_pass_count,daily_path_fold_count,sequence_path_ready,full_gate,min_relative_return_pp,...,max_sequence_avg_pos,max_sequence_max_dd_pct,max_sequence_breach_capture,mean_gate_loss,mean_drift_loss,mean_sequence_path_penalty,mean_missing_sequence_path_penalty,mean_complexity_penalty,checkpoint_penalty_loss,worst_fold_loss
0,sequence_target_cap_bc::baseline_no_sequence_cap,sequence_target_cap_bc,baseline_no_sequence_cap,bc_sequence_target_cap,2,1,2,True,False,30.501116,...,0.160863,18.396429,1.0,0.226429,0.0,0.227874,0.00,0.00,0.454302,0.908604
1,sequence_target_cap_bc::trend_break_any_cap10,sequence_target_cap_bc,trend_break_any_cap10,bc_sequence_target_cap,2,1,2,True,False,37.779333,...,0.135131,19.550984,1.0,0.303399,0.0,0.166073,0.00,0.08,0.549472,1.018944
2,checkpoint_selector::bc_initialization,checkpoint_selector,bc_initialization,bc_initialization,2,1,0,False,False,30.501116,...,0.000000,0.000000,0.0,0.230339,0.0,0.000000,0.35,0.00,0.580339,0.810678
3,checkpoint_selector::ppo_retained_iter_0_bc_eq...,checkpoint_selector,ppo_retained_iter_0_bc_equivalent,ppo_retained,2,1,0,False,False,30.501116,...,0.000000,0.000000,0.0,0.230339,0.0,0.000000,0.35,0.00,0.580339,0.810678
4,sequence_target_cap_bc::slope_low_cap15,sequence_target_cap_bc,slope_low_cap15,bc_sequence_target_cap,2,0,0,False,False,37.219027,...,0.000000,0.000000,0.0,0.474951,0.0,0.000000,0.35,0.02,0.844951,1.171159
5,sequence_target_cap_bc::trend_break_two_of_thr...,sequence_target_cap_bc,trend_break_two_of_three_cap15,bc_sequence_target_cap,2,0,0,False,False,41.768073,...,0.000000,0.000000,0.0,0.615729,0.0,0.000000,0.35,0.06,1.025729,1.156682
6,sequence_target_cap_bc::trend_break_any_cap15,sequence_target_cap_bc,trend_break_any_cap15,bc_sequence_target_cap,2,1,0,False,False,39.711227,...,0.000000,0.000000,0.0,0.634872,0.0,0.000000,0.35,0.06,1.044872,1.679744
7,sequence_target_cap_bc::anchor_any_cap15,sequence_target_cap_bc,anchor_any_cap15,bc_sequence_target_cap,2,0,0,False,False,37.183853,...,0.000000,0.000000,0.0,1.172609,0.0,0.000000,0.35,0.04,1.562609,1.707958
8,checkpoint_selector::ppo_retained_iter_4,checkpoint_selector,ppo_retained_iter_4,ppo_retained,2,1,0,False,False,41.770333,...,0.000000,0.000000,0.0,1.485015,0.0,0.000000,0.35,0.00,1.835015,3.320029
9,checkpoint_selector::ppo_selected_best,checkpoint_selector,ppo_selected_best,ppo_selected,2,1,0,False,False,41.770333,...,0.000000,0.000000,0.0,1.485015,0.0,0.000000,0.35,0.00,1.835015,3.320029


In [8]:
conclusion_lines = summary['conclusion']
conclusion_md = f"""
## Runtime conclusion

Decision: `{decision}`.

Selected candidate: `{selection.get('selected_candidate_id')}` from `{selection.get('selected_source')}`.

Key metrics:
- selected min relative return: `{selection.get('selected_min_relative_return_pp'):.2f}pp`
- selected max MaxDD: `{selection.get('selected_max_max_dd_pct'):.2f}%`
- selected penalty loss: `{selection.get('selected_checkpoint_penalty_loss'):.4f}`
- full-gate candidate count: `{selection.get('full_gate_candidate_count')}`
- robust daily full-gate candidate count: `{selection.get('robust_daily_full_gate_candidate_count')}`

Interpretation:
- The penalty selector still lands on the no-sequence-cap BC baseline, not a new PPO or target-cap variant.
- `trend_break_any_cap10` improves relative return but still leaves `wf_2022` MaxDD above the `15%` gate, so it is not a promotion or training launch point.
- Existing PPO retained checkpoints remain aggregate-only for sequence-risk paths and already fail `wf_2022` drawdown drift, so they should not be extended blindly.

Next action: `{selection.get('next_mechanism')}`.

Artifacts: `daily_sequence_risk_penalty.csv`, `metric_source_reconciliation.csv`, `fold_penalty_scores.csv`, `candidate_penalty_scores.csv`, `selection.json`, `summary.json`.
"""

display(Markdown(conclusion_md))
print('\n'.join(conclusion_lines))



## Runtime conclusion

Decision: `sequence_risk_checkpoint_penalty_confirms_no_safe_policy_improvement_no_promotion`.

Selected candidate: `sequence_target_cap_bc::baseline_no_sequence_cap` from `sequence_target_cap_bc`.

Key metrics:
- selected min relative return: `30.50pp`
- selected max MaxDD: `18.40%`
- selected penalty loss: `0.4543`
- full-gate candidate count: `0`
- robust daily full-gate candidate count: `0`

Interpretation:
- The penalty selector still lands on the no-sequence-cap BC baseline, not a new PPO or target-cap variant.
- `trend_break_any_cap10` improves relative return but still leaves `wf_2022` MaxDD above the `15%` gate, so it is not a promotion or training launch point.
- Existing PPO retained checkpoints remain aggregate-only for sequence-risk paths and already fail `wf_2022` drawdown drift, so they should not be extended blindly.

Next action: `implement_or_screen_reusable_reward_drawdown_drift_penalty_or_sequence_risk_state_feature_before_longer_ppo`.

Artifacts: `daily_sequence_risk_penalty.csv`, `metric_source_reconciliation.csv`, `fold_penalty_scores.csv`, `candidate_penalty_scores.csv`, `selection.json`, `summary.json`.


Selected sequence_target_cap_bc::baseline_no_sequence_cap with loss=0.4543, full_gate=False.
Selected max MaxDD=18.40% and min relative=30.50pp versus target <=15% / >=30pp.
Full-gate candidates=0, robust daily full-gate candidates=0.
No runtime promotion, no actor retention, and no longer PPO branch from this diagnostic card.
